In [1]:
import ydf
import pandas as pd

In [2]:
predictors = ["grid", "position", "pos_delta", "driver_code", "constructor_code", "circuit_code", "grid_rolling", "position_rolling", "pos_delta_rolling"]

In [3]:
data = pd.read_csv("./data/final_rolling.csv")

In [4]:
data.dropna(subset="position", inplace=True)

In [5]:
data["position"] = data["position"].astype("int")

In [6]:
training = data[data["raceId"] < 2022]
test = data[data["year"] >= 2022]

In [7]:
model = ydf.RandomForestLearner(label="position").train(training[predictors])

Train model on 2403 examples
Model trained in 0:00:01.218537


In [8]:
model.describe()

In [9]:
predictions = model.predict(test[predictors])

In [10]:
predictions_df = pd.DataFrame(predictions)

In [11]:
predictions_df

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.863333,0.000000,0.000000,0.000000,0.076667,0.036667,0.013333,0.006667,0.003333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.096667,0.003333,0.000000,0.000000,0.703333,0.183333,0.013333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.080000,0.000000,0.000000,0.000000,0.266667,0.516666,0.106667,0.016667,0.003333,0.010000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.010000,0.016667,0.010000,0.003333,0.000000,0.113333,0.606666,0.016667,0.116667,0.073333,0.020000,0.010000,0.000000,0.003333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.006667,0.020000,0.030000,0.016667,0.003333,0.020000,0.046667,0.440000,0.300000,0.036667,0.013333,0.033333,0.010000,0.006667,0.006667,0.003333,0.006667,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1013,0.000000,0.016667,0.016667,0.006667,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.006667,0.033333,0.123333,0.150000,0.626666,0.020000,0.000000,0.000000,0.000000
1014,0.000000,0.000000,0.000000,0.003333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.003333,0.000000,0.010000,0.200000,0.679999,0.066667,0.026667,0.010000
1015,0.000000,0.013333,0.020000,0.010000,0.000000,0.000000,0.003333,0.000000,0.003333,0.003333,0.003333,0.016667,0.023333,0.056667,0.083333,0.043333,0.100000,0.500000,0.076667,0.043333
1016,0.000000,0.006667,0.000000,0.010000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.003333,0.006667,0.033333,0.053333,0.093333,0.176667,0.496666,0.120000


In [12]:
evaluation = model.evaluate(test[predictors])

print(f"test accuracy {evaluation.accuracy}")

print("Full eval report")
evaluation

test accuracy 0.9754420432220039
Full eval report


Label \ Pred,1,10,11,12,2,3,4,5,6,7,8,9,13,14,15,16,17,18,19,20
1,57,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
10,0,58,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
11,0,0,57,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
12,0,0,0,56,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,58,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,55,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,2,57,0,0,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,58,0,0,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,1,1,0,58,0,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,57,0,0,0,0,0,0,0,0,0,0


In [13]:
model.analyze(test[predictors], sampling=0.1)

In [16]:
full_table = pd.merge(test, predictions_df, on=test.index)

In [18]:
full_table["max_pred"] = full_table[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]].max(axis=1)

In [20]:
full_table[["raceId", "driverRef", "constructorRef", "position", "circuitRef", "max_pred"]]

,raceId,driverRef,constructorRef,position,circuitRef,max_pred
0,1074,leclerc,ferrari,1,bahrain,0.863333
1,1074,sainz,ferrari,2,bahrain,0.703333
2,1074,hamilton,mercedes,3,bahrain,0.516666
3,1074,russell,mercedes,4,bahrain,0.606666
4,1074,kevin_magnussen,haas,5,bahrain,0.440000
...,...,...,...,...,...,...
1013,1134,tsunoda,rb,16,spa,0.626666
1014,1134,sargeant,williams,17,spa,0.679999
1015,1134,hulkenberg,haas,18,spa,0.500000
1016,1134,zhou,sauber,19,spa,0.496666
